In [1]:
import pandas as pd
from deep_translator import GoogleTranslator

from pathlib import Path
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import joblib

import pandas as pd

START_YEAR = 2019

In [2]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)

In [3]:
class Classifier(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [4]:
def embed(texts, batch_size=32):
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, return_tensors='pt').to(device)
            output = model(**encoded)
            cls_embeddings = output.last_hidden_state[:, 0, :]  # [CLS] token
            embeddings.append(cls_embeddings.cpu())
    return torch.cat(embeddings)

CIHR

In [5]:
cihr_path = "raw_data/CIHR/"
cihr_files = Path(cihr_path).glob("*.csv")

CIHR_DFS = [pd.read_csv(f) for f in cihr_files]
CIHR_DATA = pd.concat(CIHR_DFS, ignore_index=True)

In [6]:
grant_descriptors = [
    "ApplicationTitle_TitreDemande", "PrimaryThemeEN_ThemePrincipalAN", "AllResearchCategoriesEN_TousCategoriesRechercheAN", "ApplicationKeywords_MotsClesDemande"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

CIHR_DATA = CIHR_DATA[grant_descriptors]
CIHR_DATA.columns = col_names

CIHR_DATA.drop_duplicates(inplace=True)
CIHR_DATA["Main_Discipline"].value_counts()

Main_Discipline
Biomedical                                         8989
Clinical                                           3869
Social/Cultural/Environmental/Population Health    3003
Health systems/services                            2830
Not applicable/Specified                            127
Name: count, dtype: int64

Training CIHR Main Discipline

In [7]:
tmp_data = CIHR_DATA.sample(frac=1).reset_index(drop=True) # shuffle

# tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data[tmp_data["Main_Discipline"] != "Not applicable/Specified"]

tmp_data = tmp_data.dropna(subset=['Main_Discipline'])
tmp_data["Main_Discipline"].value_counts()


Main_Discipline
Biomedical                                         8989
Clinical                                           3869
Social/Cultural/Environmental/Population Health    3003
Health systems/services                            2830
Name: count, dtype: int64

In [8]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [9]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 249.5628
Epoch 2: Loss = 181.0797
Epoch 3: Loss = 164.7086
Epoch 4: Loss = 158.7684
Epoch 5: Loss = 155.1246
Epoch 6: Loss = 152.7117
Epoch 7: Loss = 151.0238
Epoch 8: Loss = 149.4899
Epoch 9: Loss = 148.0713
Epoch 10: Loss = 146.5656
Epoch 11: Loss = 145.2145
Epoch 12: Loss = 144.2916
Epoch 13: Loss = 143.4599
Epoch 14: Loss = 142.7414
Epoch 15: Loss = 141.6481
Epoch 16: Loss = 141.5368
Epoch 17: Loss = 140.0895
Epoch 18: Loss = 139.5964
Epoch 19: Loss = 138.3035
Epoch 20: Loss = 138.0758
Epoch 21: Loss = 137.1057
Epoch 22: Loss = 136.4804
Epoch 23: Loss = 135.5043
Epoch 24: Loss = 134.8138
Epoch 25: Loss = 134.0210
Epoch 26: Loss = 133.6206
Epoch 27: Loss = 132.2431
Epoch 28: Loss = 132.1895
Epoch 29: Loss = 131.2812
Epoch 30: Loss = 131.2000
Epoch 31: Loss = 130.0517
Epoch 32: Loss = 129.4002
Epoch 33: Loss = 128.6922
Epoch 34: Loss = 128.4671
Epoch 35: Loss = 127.1645
Epoch 36: Loss = 126.7421
Epoch 37: Loss = 125.3457
Epoch 38: Loss = 125.5260
Epoch 39: Loss = 124.

In [10]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()

print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                 precision    recall  f1-score   support

                                     Biomedical       0.88      0.91      0.89      1798
                                       Clinical       0.61      0.60      0.60       774
                        Health systems/services       0.61      0.59      0.60       566
Social/Cultural/Environmental/Population Health       0.68      0.63      0.66       601

                                       accuracy                           0.75      3739
                                      macro avg       0.69      0.68      0.69      3739
                                   weighted avg       0.75      0.75      0.75      3739



In [11]:
torch.save(clf_model.state_dict(), "models/CIHR_MD.pt")
joblib.dump(label_mapping, "models/CIHR_MD_label_mapping.pkl")


['models/CIHR_MD_label_mapping.pkl']

Training CIHR Area of Research

In [12]:
tmp_data = CIHR_DATA.sample(frac=1).reset_index(drop=True) # shuffle

# tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data[tmp_data["Area_of_Research"] != "Not applicable/Specified"]

tmp_data = tmp_data.dropna(subset=['Area_of_Research'])
tmp_data["Area_of_Research"].value_counts()

Area_of_Research
Health; Other medical research                                                                                                                                              127
Cancer-Cancer Drug Development and Therapeutics                                                                                                                             109
Cardiovascular, Respiratory and Circulatory Systems-Cardiovascular and Circulatory Sciences                                                                                 108
HEALTH SERVICES RESEARCH                                                                                                                                                     89
Health; Medical research, hospital treatment, surgery                                                                                                                        81
                                                                                                       

NSERC

In [13]:
nserc_path = "raw_data/NSERC/"
nserc_files = Path(nserc_path).glob("*.csv")

NSERC_DFS = [pd.read_csv(f) for f in nserc_files]
NSERC_DATA = pd.concat(NSERC_DFS, ignore_index=True)

In [14]:
grant_descriptors = [
    "ApplicationTitle", "AreaOfApplicationGroupEN", "ResearchSubjectEN", "Keyword"
]

col_names = [
     'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]

NSERC_DATA = NSERC_DATA[grant_descriptors]
NSERC_DATA.columns = col_names

NSERC_DATA.drop_duplicates(inplace=True)

Model for Main Discipline

In [15]:
tmp_data = NSERC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

tmp_data = tmp_data[tmp_data["Main_Discipline"] != "Not available"]

tmp_data["Main_Discipline"].value_counts()

Main_Discipline
Advancement of knowledge                   20177
Manufacturing processes and products        4626
Information and communication services      3936
Environment                                 3927
Energy resources                            3123
Health, education and social services       2937
Transportation systems and services         2077
Agriculture and primary food production     1541
Construction, urban and rural planning      1376
Northern development                        1203
Natural resources (economic aspects)        1106
The socioeconomic objective available        737
Commercial services                          400
Name: count, dtype: int64

In [16]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.tolist())
X_test = embed(X_test_text.tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [17]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 1060.2804
Epoch 2: Loss = 813.6243
Epoch 3: Loss = 768.6505
Epoch 4: Loss = 747.9912
Epoch 5: Loss = 735.6809
Epoch 6: Loss = 726.1273
Epoch 7: Loss = 717.7575
Epoch 8: Loss = 710.5404
Epoch 9: Loss = 704.6491
Epoch 10: Loss = 697.9206
Epoch 11: Loss = 693.4207
Epoch 12: Loss = 688.3468
Epoch 13: Loss = 681.8113
Epoch 14: Loss = 678.3727
Epoch 15: Loss = 672.8097
Epoch 16: Loss = 667.2427
Epoch 17: Loss = 663.9415
Epoch 18: Loss = 659.0041
Epoch 19: Loss = 655.3313
Epoch 20: Loss = 652.8665
Epoch 21: Loss = 646.6907
Epoch 22: Loss = 643.3811
Epoch 23: Loss = 638.5588
Epoch 24: Loss = 635.7424
Epoch 25: Loss = 632.1840
Epoch 26: Loss = 627.6902
Epoch 27: Loss = 625.0741
Epoch 28: Loss = 620.5793
Epoch 29: Loss = 615.9400
Epoch 30: Loss = 612.7957
Epoch 31: Loss = 609.3452
Epoch 32: Loss = 606.5329
Epoch 33: Loss = 602.2197
Epoch 34: Loss = 599.0560
Epoch 35: Loss = 595.9086
Epoch 36: Loss = 592.7330
Epoch 37: Loss = 589.1359
Epoch 38: Loss = 586.0915
Epoch 39: Loss = 581

In [18]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()

print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                         precision    recall  f1-score   support

               Advancement of knowledge       0.81      0.91      0.86      4036
Agriculture and primary food production       0.82      0.67      0.73       308
                    Commercial services       0.77      0.62      0.69        80
 Construction, urban and rural planning       0.80      0.78      0.79       275
                       Energy resources       0.83      0.81      0.82       625
                            Environment       0.82      0.76      0.79       786
  Health, education and social services       0.79      0.71      0.75       588
 Information and communication services       0.85      0.79      0.82       787
   Manufacturing processes and products       0.68      0.69      0.68       925
   Natural resources (economic aspects)       0.85      0.66      0.74       221
                   Northern development       0.78      0.63      0.70       241
  The socioeconomic objecti

In [19]:
torch.save(clf_model.state_dict(), "models/NSERC_MD.pt")
joblib.dump(label_mapping, "models/NSERC_MD_label_mapping.pkl")

['models/NSERC_MD_label_mapping.pkl']

In [20]:
tmp_data = NSERC_DATA.sample(frac=1).reset_index(drop=True) # shuffle
translator = GoogleTranslator(target="en")
                              
val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 50].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

for class_name in classes:
    tmp_data["Area_of_Research"] = tmp_data["Area_of_Research"].replace(class_name, translator.translate(class_name))

val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 100].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

tmp_data['Area_of_Research'] = tmp_data['Area_of_Research'].str.replace(r' \(.*?\)', '', regex=True)
                              
tmp_data = tmp_data.groupby(["Area_of_Research"]).head(500)
tmp_data = tmp_data[tmp_data["Area_of_Research"] != "Not available"]

tmp_data = tmp_data.dropna(subset=['Area_of_Research'])
tmp_data["Area_of_Research"].value_counts()

Area_of_Research
Fluid mechanics                   500
Neurophysiology                   500
Inorganic chemistry               500
Analytical chemistry              500
Statistics and probability        500
                                 ... 
Animal nutrition and husbandry    105
Photonics                         104
Fuel and energy technology        102
Enzymes                           102
Database management               101
Name: count, Length: 135, dtype: int64

In [21]:
label_mapping = dict(enumerate(tmp_data['Area_of_Research'].astype('category').cat.categories))
tmp_data['Area_of_Research'] = tmp_data['Area_of_Research'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Area_of_Research'], test_size=0.2, stratify=tmp_data['Area_of_Research'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [22]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 2221.5352
Epoch 2: Loss = 1876.6517
Epoch 3: Loss = 1632.0547
Epoch 4: Loss = 1500.1016
Epoch 5: Loss = 1431.1785
Epoch 6: Loss = 1384.6375
Epoch 7: Loss = 1352.1158
Epoch 8: Loss = 1327.2465
Epoch 9: Loss = 1305.2663
Epoch 10: Loss = 1288.6731
Epoch 11: Loss = 1276.3502
Epoch 12: Loss = 1262.6036
Epoch 13: Loss = 1252.9321
Epoch 14: Loss = 1243.4973
Epoch 15: Loss = 1233.2153
Epoch 16: Loss = 1227.0588
Epoch 17: Loss = 1217.1627
Epoch 18: Loss = 1213.2481
Epoch 19: Loss = 1204.7984
Epoch 20: Loss = 1199.7152
Epoch 21: Loss = 1193.1257
Epoch 22: Loss = 1184.4969
Epoch 23: Loss = 1177.7173
Epoch 24: Loss = 1175.3359
Epoch 25: Loss = 1169.1479
Epoch 26: Loss = 1163.9325
Epoch 27: Loss = 1159.4384
Epoch 28: Loss = 1154.2832
Epoch 29: Loss = 1152.3088
Epoch 30: Loss = 1146.4052
Epoch 31: Loss = 1146.3671
Epoch 32: Loss = 1141.0097
Epoch 33: Loss = 1136.0808
Epoch 34: Loss = 1129.6971
Epoch 35: Loss = 1127.2785
Epoch 36: Loss = 1124.6897
Epoch 37: Loss = 1118.7819
Epoch 38: 

In [23]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()
    
print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                            precision    recall  f1-score   support

                                    Advanced manufacturing       0.26      0.25      0.25        32
        Aerospace, aeronautical and automotive engineering       0.37      0.39      0.38        92
                                  Agricultural engineering       0.48      0.45      0.47        22
                                                Algorithms       0.42      0.29      0.34        34
                                      Analytical chemistry       0.32      0.41      0.36       100
                                            Animal biology       0.12      0.11      0.11       100
                                            Animal ecology       0.36      0.44      0.39       100
                            Animal nutrition and husbandry       0.56      0.43      0.49        21
                          Animal physiology and metabolism       0.37      0.42      0.39       100

In [24]:
torch.save(clf_model.state_dict(), "models/NSERC_AR.pt")
joblib.dump(label_mapping, "models/NSERC_AR_label_mapping.pkl")

['models/NSERC_AR_label_mapping.pkl']

SSHRC

In [25]:
sshrc_path = "raw_data/SSHRC/"
sshrc_files = Path(sshrc_path).glob("*.csv")

SSHRC_DFS = [pd.read_csv(f) for f in sshrc_files]
SSHRC_DATA = pd.concat(SSHRC_DFS, ignore_index=True)

In [26]:
grant_descriptors = [
    "Title-Titre", "Main_Discipline", "Area_of_Research", "Keywords-Mots-clés"
]

col_names = [
    'Title', 'Main_Discipline', 'Area_of_Research', 'Keywords'
]


SSHRC_DATA = SSHRC_DATA[grant_descriptors]
SSHRC_DATA.columns = col_names

SSHRC_DATA.drop_duplicates(inplace=True)

In [27]:
SSHRC_DATA["Main_Discipline"].value_counts()

tmp_data = SSHRC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

val_counts = tmp_data["Main_Discipline"].value_counts()
classes = val_counts[val_counts > 200].index
tmp_data = tmp_data[tmp_data["Main_Discipline"].isin(classes)]

tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data.dropna(subset=['Main_Discipline'])
tmp_data = tmp_data[~(tmp_data['Main_Discipline'].isin(["Not Specified", "Not specified", "Not Applicable", "Multiple primary fields of research", "Interdisciplinary Studies"]))]


tmp_data["Main_Discipline"].value_counts()


Main_Discipline
Psychology                                           500
Urban and Regional Studies, Environmental Studies    500
Anthropology                                         500
Linguistics                                          500
Communications and Media Studies                     500
Social Work                                          500
History                                              500
Philosophy                                           500
Sociology                                            500
Law                                                  500
Education                                            500
Economics                                            500
Management, Business, Administrative Studies         500
Fine Arts                                            500
Literature, Modern Languages and                     500
Political Science                                    500
Geography                                            500
Criminology    

In [28]:
label_mapping = dict(enumerate(tmp_data['Main_Discipline'].astype('category').cat.categories))
tmp_data['Main_Discipline'] = tmp_data['Main_Discipline'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Main_Discipline'], test_size=0.2, stratify=tmp_data['Main_Discipline'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [29]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 360.5987
Epoch 2: Loss = 344.2516
Epoch 3: Loss = 320.5325
Epoch 4: Loss = 294.5783
Epoch 5: Loss = 271.9552
Epoch 6: Loss = 256.0170
Epoch 7: Loss = 243.5544
Epoch 8: Loss = 234.7252
Epoch 9: Loss = 227.9337
Epoch 10: Loss = 223.6382
Epoch 11: Loss = 218.1023
Epoch 12: Loss = 214.7461
Epoch 13: Loss = 211.7976
Epoch 14: Loss = 208.9636
Epoch 15: Loss = 206.3646
Epoch 16: Loss = 204.5922
Epoch 17: Loss = 203.5573
Epoch 18: Loss = 200.8892
Epoch 19: Loss = 199.1948
Epoch 20: Loss = 198.6377
Epoch 21: Loss = 196.5100
Epoch 22: Loss = 195.2530
Epoch 23: Loss = 193.7236
Epoch 24: Loss = 192.0722
Epoch 25: Loss = 191.8061
Epoch 26: Loss = 190.6121
Epoch 27: Loss = 190.0896
Epoch 28: Loss = 188.5708
Epoch 29: Loss = 187.6959
Epoch 30: Loss = 186.1663
Epoch 31: Loss = 185.4013
Epoch 32: Loss = 184.9031
Epoch 33: Loss = 184.2091
Epoch 34: Loss = 182.7815
Epoch 35: Loss = 181.9750
Epoch 36: Loss = 182.3311
Epoch 37: Loss = 180.8005
Epoch 38: Loss = 179.2994
Epoch 39: Loss = 179.

In [30]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()
    
print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                   precision    recall  f1-score   support

                                     Anthropology       0.36      0.34      0.35       100
                                      Archaeology       0.66      0.62      0.64        81
                 Communications and Media Studies       0.43      0.44      0.44       100
                                      Criminology       0.57      0.63      0.60        93
                                        Economics       0.50      0.54      0.52       100
                                        Education       0.61      0.62      0.62       100
                                        Fine Arts       0.43      0.46      0.44       100
                                        Geography       0.43      0.38      0.40       100
                                          History       0.42      0.52      0.47       100
                                              Law       0.57      0.55      0.56       10

In [31]:
torch.save(clf_model.state_dict(), "models/SSHRC_MD.pt")
joblib.dump(label_mapping, "models/SSHRC_MD_label_mapping.pkl")

['models/SSHRC_MD_label_mapping.pkl']

In [32]:
tmp_data = SSHRC_DATA.sample(frac=1).reset_index(drop=True) # shuffle

# tmp_data = tmp_data.groupby(["Main_Discipline"]).head(500)
tmp_data = tmp_data[~(tmp_data['Area_of_Research'].isin(["Not Specified", "Not Subject to Research Classification"]))]

val_counts = tmp_data["Area_of_Research"].value_counts()
classes = val_counts[val_counts > 100].index
tmp_data = tmp_data[tmp_data["Area_of_Research"].isin(classes)]

tmp_data = tmp_data.dropna(subset=['Area_of_Research'])
tmp_data = tmp_data.groupby(["Area_of_Research"]).head(500)
tmp_data["Area_of_Research"].value_counts()

Area_of_Research
Indigenous peoples                                      500
Science and technology                                  500
Health                                                  500
Multiculturalism and ethnic studies                     500
Violence                                                500
Management                                              500
Youth                                                   500
Women                                                   500
Employment and labour                                   500
Mental Health                                           500
Law and Justice                                         500
Post-Secondary Education and Research                   500
Social development and welfare                          500
Immigration                                             500
Environment and Sustainability                          500
Children                                                500
Education              

In [33]:
label_mapping = dict(enumerate(tmp_data['Area_of_Research'].astype('category').cat.categories))
tmp_data['Area_of_Research'] = tmp_data['Area_of_Research'].astype('category').cat.codes

X_train_text, X_test_text, y_train, y_test = train_test_split(
    tmp_data['Title'], tmp_data['Area_of_Research'], test_size=0.2, stratify=tmp_data['Area_of_Research'], #random_state=42
)

X_train = embed(X_train_text.astype(str).tolist())
X_test = embed(X_test_text.astype(str).tolist())
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)

input_dim = X_train.shape[1]
output_dim = len(label_mapping)

clf_model = Classifier(input_dim, output_dim).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(clf_model.parameters(), lr=1e-4)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [34]:
# Train
for epoch in range(500):
    clf_model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = clf_model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 724.2370
Epoch 2: Loss = 669.8976
Epoch 3: Loss = 607.0764
Epoch 4: Loss = 555.5589
Epoch 5: Loss = 518.1603
Epoch 6: Loss = 494.8621
Epoch 7: Loss = 478.0982
Epoch 8: Loss = 465.4020
Epoch 9: Loss = 456.5665
Epoch 10: Loss = 447.8445
Epoch 11: Loss = 443.0708
Epoch 12: Loss = 437.7184
Epoch 13: Loss = 431.8901
Epoch 14: Loss = 428.5257
Epoch 15: Loss = 426.5867
Epoch 16: Loss = 422.5966
Epoch 17: Loss = 419.5029
Epoch 18: Loss = 417.7070
Epoch 19: Loss = 414.3392
Epoch 20: Loss = 410.8513
Epoch 21: Loss = 411.2790
Epoch 22: Loss = 409.6235
Epoch 23: Loss = 406.3598
Epoch 24: Loss = 403.3830
Epoch 25: Loss = 402.7533
Epoch 26: Loss = 400.4573
Epoch 27: Loss = 398.5731
Epoch 28: Loss = 396.4550
Epoch 29: Loss = 395.2039
Epoch 30: Loss = 393.1570
Epoch 31: Loss = 392.4857
Epoch 32: Loss = 390.6223
Epoch 33: Loss = 389.8682
Epoch 34: Loss = 387.9508
Epoch 35: Loss = 386.5655
Epoch 36: Loss = 385.8863
Epoch 37: Loss = 385.2553
Epoch 38: Loss = 383.3974
Epoch 39: Loss = 381.

In [35]:
clf_model.eval()
with torch.no_grad():
    preds = clf_model(X_test.to(device))
    pred_labels = preds.argmax(dim=1).cpu().numpy()
    true_labels = y_test.numpy()
    
print(classification_report(true_labels, pred_labels, target_names=list(label_mapping.values()), zero_division=0))

                                                      precision    recall  f1-score   support

                                         Agriculture       0.39      0.38      0.38        37
                                    Arts and culture       0.40      0.46      0.43       100
                                            Children       0.53      0.54      0.54       100
                                  Children and youth       0.11      0.04      0.06        24
                                       Communication       0.33      0.34      0.34       100
                   Economic and Regional Development       0.38      0.38      0.38        95
                                           Education       0.43      0.43      0.43       100
                                             Elderly       0.75      0.63      0.69        68
                               Employment and labour       0.50      0.51      0.50       100
                        Energy and natural resources       

In [36]:
torch.save(clf_model.state_dict(), "models/SSHRC_AR.pt")
joblib.dump(label_mapping, "models/SSHRC_AR_label_mapping.pkl")

['models/SSHRC_AR_label_mapping.pkl']